# SAM3 Zero-Shot Tomato Ripeness — Direct Inference

No training. No annotations. Just point SAM3 at your images.

**Requires:**
- HuggingFace access approved at https://huggingface.co/facebook/sam3
- Kaggle secret: `HF_TOKEN`
- GPU: ON (T4)

In [ ]:
!pip install -q ultralytics huggingface_hub

import os, glob, random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import torch
from pathlib import Path
from huggingface_hub import hf_hub_download, login
from ultralytics.models.sam import SAM3SemanticPredictor
from IPython.display import Image, display

print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
# Download SAM3
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret('HF_TOKEN'))

SAM3_PATH = hf_hub_download(
    repo_id='facebook/sam3',
    filename='sam3.pt',
    local_dir='/kaggle/working',
)
print(f'SAM3: {SAM3_PATH}  ({os.path.getsize(SAM3_PATH)/1e6:.0f} MB)')

In [ ]:
# Load SAM3
predictor = SAM3SemanticPredictor(overrides=dict(
    conf=0.30,
    task='segment',
    mode='predict',
    model=SAM3_PATH,
    half=True,   # FP16 — faster on T4, no accuracy loss
))

# Class prompts — tweak these if results are off
# (e.g. cherry tomatoes: "small unripe cherry tomato")
TEXT_PROMPTS = [
    'unripe green tomato',       # class 0: Unripe
    'half ripe orange tomato',   # class 1: Half_Ripe
    'ripe red tomato',           # class 2: Ripe
]
CLASS_NAMES  = ['Unripe', 'Half_Ripe', 'Ripe']
CLASS_COLORS = ['#2ecc71', '#f39c12', '#e74c3c']

print('SAM3 ready.')

In [ ]:
# Inference helper
def predict_ripeness(image_path, conf_threshold=0.30):
    """
    Run SAM3 on a single image and return detections.
    Returns list of (class_id, confidence, [x1,y1,x2,y2]) tuples.
    """
    predictor.set_image(str(image_path))
    detections = []

    for cls_id, prompt in enumerate(TEXT_PROMPTS):
        results = predictor(text=[prompt])
        for result in results:
            if result.boxes is None or len(result.boxes) == 0:
                continue
            for i in range(len(result.boxes)):
                conf = float(result.boxes.conf[i].cpu())
                if conf < conf_threshold:
                    continue
                box = result.boxes.xyxy[i].cpu().numpy().tolist()
                detections.append((cls_id, conf, box))

    return detections


def visualize(image_path, detections, save_path=None):
    img = plt.imread(str(image_path))
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(img)

    counts = {n: 0 for n in CLASS_NAMES}
    for cls_id, conf, (x1, y1, x2, y2) in detections:
        color = CLASS_COLORS[cls_id]
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, f'{CLASS_NAMES[cls_id]} {conf:.2f}',
                color=color, fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
        counts[CLASS_NAMES[cls_id]] += 1

    title = '  |  '.join(f'{k}: {v}' for k, v in counts.items())
    ax.set_title(f'{Path(image_path).name}\n{title}', fontsize=11)
    ax.axis('off')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    return counts


print('Helper functions ready.')

In [ ]:
# --- Run on your images ---
# Update IMAGES_DIR to your Kaggle input dataset
IMAGES_DIR = '/kaggle/input/my-tomato-images'   # <-- update this

IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
all_images = [
    p for p in Path(IMAGES_DIR).rglob('*')
    if p.suffix.lower() in IMG_EXTENSIONS
]
print(f'Found {len(all_images)} images')

os.makedirs('/kaggle/working/sam3_output', exist_ok=True)

for img_path in all_images:
    detections = predict_ripeness(img_path, conf_threshold=0.30)
    save_path = f'/kaggle/working/sam3_output/{img_path.stem}_result.png'
    counts = visualize(img_path, detections, save_path=save_path)
    print(f'{img_path.name}: {len(detections)} tomatoes — {counts}')

In [ ]:
# --- Tune confidence if needed ---
#
# Too many false positives (wall, leaves being detected)?
#   → Raise conf_threshold: 0.35, 0.40, 0.45
#
# Missing obvious tomatoes?
#   → Lower conf_threshold: 0.25, 0.20
#   → Also try more specific prompts:
#       'small green unripe tomato on a plant'
#       'fully red ripe tomato'
#
# Half_Ripe and Ripe getting confused?
#   → This is the known weakness of zero-shot SAM3 for fine-grained ripeness.
#   → If this matters, go back to sam3_ripeness_pipeline.ipynb which fine-tunes
#   → a YOLO model on SAM3 auto-labels from YOUR images.

# Quick single-image test with different thresholds
test_image = all_images[0]   # change index to test different images

for threshold in [0.20, 0.30, 0.40, 0.50]:
    dets = predict_ripeness(test_image, conf_threshold=threshold)
    print(f'conf={threshold}: {len(dets)} detections')